# Taxa analysis preparation

**Purpose:** Build the single analysis-ready taxa handoff used by all taxa results notebooks.

This notebook owns source linkage, serialized-label parsing, broad/analysis/detail taxonomic states, match accounting, and the GBIF described-diversity benchmark. The preserved grain is exactly **one row per publication (`UT`)**; list-valued labels remain nested rather than being exploded.

Only three durable artifacts are produced: one publication Parquet, one compact GBIF benchmark CSV, and one provenance/audit manifest. Results notebooks perform their own analytical filtering, estimation, and plotting from this bundle.

In [1]:
import sys
from pathlib import Path

import pandas as pd

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "data_helpers").is_dir() and (candidate / "checklists").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the biodiversity repository root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_helpers import results_config
from data_helpers.prep import corpus
from data_helpers.prep import taxa_analysis_prep


In [2]:
RESULTS_CONFIG = results_config.load_results_config()
CONFIG = RESULTS_CONFIG["taxa_analysis_prep"]
DRIVER_ORDER, _, _ = results_config.driver_l1_display(RESULTS_CONFIG)
THREAT_ORDER, _, _ = results_config.threat_l0_display(RESULTS_CONFIG)
MAPPING_PATH = PROJECT_ROOT / CONFIG["taxa_group_mapping"]
MAPPING = taxa_analysis_prep.load_taxa_mapping(MAPPING_PATH)
STORE = taxa_analysis_prep.TaxaPreparedStore(
    PROJECT_ROOT / CONFIG["output_directory"]
)


## 1. Build the publication-level handoff

The configured eligible corpus already enforces one row per `UT` across the coding sources. Taxa enrichment is joined one-to-one and the original key order must remain unchanged. Driver, Threat-L0, realm, and all three taxonomic resolutions are retained as ordered list columns.

In [3]:
corpus_df = corpus.build_merged_corpus()
articles, audits = taxa_analysis_prep.build_taxa_publications(
    corpus_df,
    taxa_path=PROJECT_ROOT / CONFIG["taxa_source"],
    mapping=MAPPING,
    driver_order=DRIVER_ORDER,
    threat_order=THREAT_ORDER,
)

display(pd.DataFrame(audits["source_grain"]).style.format(
    {"rows": "{:,}", "unique_UT": "{:,}", "missing_UT": "{:,}", "duplicated_UT": "{:,}"}
).hide(axis="index"))
print(f"Prepared publication grain: {len(articles):,} rows / {articles['UT'].nunique():,} unique UTs.")


source,rows,unique_UT,missing_UT,duplicated_UT,grain
eligible merged corpus,"605,199","605,199",0,0,one row per UT
taxa-with-api,"605,199","605,199",0,0,one row per UT


Prepared publication grain: 605,199 rows / 605,199 unique UTs.


In [4]:
broad_accounting = (
    articles["taxa_broad_inclusion_state"]
    .value_counts()
    .reindex(taxa_analysis_prep.INCLUSION_ORDER, fill_value=0)
    .rename_axis("taxa_broad_inclusion_state")
    .rename("n_publications")
    .reset_index()
)
broad_accounting["share_of_all_publications_pct"] = broad_accounting["n_publications"] / len(articles) * 100
display(broad_accounting.style.format({"n_publications": "{:,}", "share_of_all_publications_pct": "{:.2f}%"}).hide(axis="index"))
display(pd.DataFrame(audits["match_status"]).style.format({"taxon_item_count": "{:,}", "share_of_taxon_items_pct": "{:.2f}%"}).hide(axis="index"))


taxa_broad_inclusion_state,n_publications,share_of_all_publications_pct
Included: at least one benchmarkable broad group,"444,800",73.50%
Excluded: Not applicable only,"119,503",19.75%
Excluded: Unclear only,"4,501",0.74%
Excluded: mixed Not applicable and Unclear,0,0.00%
Excluded: accepted taxon but broad hierarchy unresolved,"4,446",0.73%
Excluded: reported taxon but no accepted benchmarkable match,"31,949",5.28%
Excluded: no taxon item returned,0,0.00%


match_status,taxon_item_count,share_of_taxon_items_pct,eligible_for_grouping
exact,"815,975",77.94%,True
special_value,"128,841",12.31%,False
rank_mismatch,"41,727",3.99%,False
ambiguous,"24,059",2.30%,False
no_match,"22,087",2.11%,False
higher_rank,"10,695",1.02%,False
fuzzy_accepted,"3,019",0.29%,True
fuzzy_review,565,0.05%,False


## 2. Build the fixed GBIF broad-group benchmark

The curated GBIF backbone is streamed once. Only accepted species-rank records enter; the current broad hierarchy rules are applied, unresolved records are excluded, and benchmarkable shares are renormalized.

In [5]:
benchmark = taxa_analysis_prep.build_gbif_broad_benchmark(
    PROJECT_ROOT / CONFIG["gbif_source"],
    MAPPING_PATH,
)
display(benchmark.audit.style.format({"records": "{:,}"}).hide(axis="index"))
display(benchmark.counts.style.format({"described_species_count": "{:,}", "described_species_share_pct": "{:.2f}%"}).hide(axis="index"))


metric,value
GBIF backbone rows scanned,7694321
Species-rank rows,4961031
Accepted species-rank rows,2598208
Accepted species rows missing taxonID,0
Broad-group benchmark total,2598208


broad_group,described_species_count,described_species_share,described_species_share_pct
Vertebrates,"119,596",0.046030,4.60%
Invertebrates,"1,683,461",0.647932,64.79%
Plants,"443,824",0.170819,17.08%
Fungi,"171,668",0.066072,6.61%
Other,"172,283",0.066308,6.63%
Unresolved,"7,376",0.002839,0.28%


## 3. Write and verify the canonical bundle

The manifest records source signatures, the semantic grouping-rule fingerprint, row reconciliation, all compact processing audits, and benchmark construction counts. A reload immediately verifies schema, rules, key grain, and artifact row counts.

In [6]:
match_summary = STORE.write_matches(
    PROJECT_ROOT / CONFIG["taxa_lineage_source"],
    expected_uts=articles["UT"],
)
sources = {
    "corpus_config": PROJECT_ROOT / CONFIG["corpus_config"],
    "taxa": PROJECT_ROOT / CONFIG["taxa_source"],
    "taxa_lineage": PROJECT_ROOT / CONFIG["taxa_lineage_source"],
    "gbif": PROJECT_ROOT / CONFIG["gbif_source"],
    "gbif_eml": PROJECT_ROOT / CONFIG["gbif_eml_source"],
    "taxa_group_mapping": MAPPING_PATH,
}
manifest = taxa_analysis_prep.build_manifest(
    articles, benchmark, mapping=MAPPING, audits=audits, match_summary=match_summary, sources=sources, repository_root=PROJECT_ROOT
)
bundle = taxa_analysis_prep.TaxaPreparedBundle(articles, benchmark, manifest)
written = STORE.write(bundle)
reloaded = STORE.load(mapping=MAPPING)
assert reloaded.articles["UT"].equals(articles["UT"])
pd.testing.assert_frame_equal(reloaded.benchmark.counts, benchmark.counts, check_exact=False, rtol=1e-12, atol=1e-12)
print("Verified canonical bundle:")
for path in written:
    print(f"  {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size / 1024**2:.2f} MiB)")


Verified canonical bundle:
  notebooks/data_processing/outputs/02_taxa_analysis_prep/taxa_publications.parquet (15.17 MiB)
  notebooks/data_processing/outputs/02_taxa_analysis_prep/taxa_matches.parquet (54.47 MiB)
  notebooks/data_processing/outputs/02_taxa_analysis_prep/gbif_broad_benchmark.csv (0.00 MiB)
  notebooks/data_processing/outputs/02_taxa_analysis_prep/manifest.json (0.03 MiB)


## Handoff contract

Downstream taxa results load publication summaries through `TaxaPreparedStore.load`. The separate `taxa_matches.parquet` stores one typed nested match list per UT, including full accepted lineage, without changing the publication grain. If taxonomic grouping semantics change, the fingerprint check requires this notebook to be rerun. Changing only plotting colors does not invalidate the prepared evidence.